In [0]:
# =============================================================================
# NB13 — Within-paper analysis: does composition explain WHICH material is
#        better when experimental conditions are held fixed?
# =============================================================================
# Context (NB11/NB12): across papers, no material representation adds signal
# beyond experimental conditions (composition is confounded with paper).
# Decision (2026-10-14): keep the absolute target as the across-paper
# benchmark AND add this within-paper analysis as the test of the
# composition effect.
#
# Design
#   Two comparison levels:
#     strict : block = DOI + identical reported conditions (run type, scavenger,
#              cocatalyst type and loading, cutoff, lamp power, catalyst mass,
#              volume, organic fraction, reaction T). Materials in a block
#              were tested under the same reported conditions.
#     paper  : block = DOI only; within-paper differences in numeric
#              experimental variables enter as (demeaned) covariates.
#   Rows are first averaged per (block, base_compound); only blocks with
#   ≥ 2 distinct compounds are kept.
#
#   Within ("fixed-effects") transformation: target and features are
#   demeaned inside each block. The model learns how DIFFERENCES between
#   materials translate into DIFFERENCES in log rate. Paper-level offsets
#   (reactor, lamp, lab) cancel out by construction.
#
#   Validation: 5×5 repeated GroupKFold by DOI (unseen papers in validation).
#   Metrics on validation blocks:
#     r2_within  — R² of block-demeaned predictions vs block-demeaned y
#                  (null model = 0)
#     pair_acc   — fraction of material pairs inside a block, with
#                  |Δy| ≥ PAIR_MIN log units, ranked in the correct order
#                  (null = 0.5)
#   Significance: permutation test. y is permuted independently inside every
#   block (N_PERM times); the full CV is re-run on repeat-0 splits to build
#   the null distribution of mean pair_acc / r2_within; p = (1 + #null ≥ obs)
#   / (1 + N_PERM). NOTE: the spread over repeated splits shown in the summary
#   does NOT include data-sampling variance (validated on synthetic data: a
#   single pure-noise dataset gives pair_acc anywhere in ~0.40–0.60), so
#   conclusions must rest on the permutation p-value and the cluster-robust
#   OLS, not on the split-percentile band.
#
#   Inference (descriptive, not predictive): within-block OLS on a few
#   z-scored physical descriptors, cluster-robust SE by DOI. Coefficient =
#   change in log10 rate per 1 SD of the descriptor, conditions fixed.
#   Implemented in numpy (CR1 sandwich, t with G-1 df) — no statsmodels
#   dependency, since pip installs do not persist on Community Edition.
#
# Inputs:  master, X_material, X_exp_v2
# Outputs: ml_perovskites.within_nb13, /tmp/nb13_within.png
# Rules:   imputation/scaling fitted on training rows only; SEED = 42
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import subprocess, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.impute        import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model  import RidgeCV
from pyspark.sql import SparkSession

SEED, SCHEMA, N_FOLDS, N_REPEATS = 42, "ml_perovskites", 5, 5
PAIR_MIN = 0.2   # log10 units (~1.6×); smaller differences treated as ties
N_PERM   = 60    # permutations for the significance test (p resolution 1/61)

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "xgboost"])
    import xgboost as xgb

try:
    from scipy import stats as _st
    def _t_sf2(t, df):            # two-sided p-value
        return 2 * _st.t.sf(np.abs(t), df)
    def _t_q(df):                 # 97.5% quantile
        return _st.t.ppf(0.975, df)
except ImportError:               # normal approximation fallback
    from math import erfc, sqrt
    def _t_sf2(t, df):
        return np.array([erfc(abs(v) / sqrt(2)) for v in np.atleast_1d(t)])
    def _t_q(df):
        return 1.96

def ols_cluster(y, X, groups):
    """OLS without intercept (data already block-demeaned) with CR1
    cluster-robust covariance — same estimator as statsmodels
    cov_type='cluster'. Inference uses t with G-1 degrees of freedom."""
    n, k = X.shape
    XtX_inv = np.linalg.pinv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    g_ids = np.unique(groups)
    G = len(g_ids)
    meat = np.zeros((k, k))
    for g in g_ids:
        s = X[groups == g].T @ e[groups == g]
        meat += np.outer(s, s)
    c = (G / (G - 1)) * ((n - 1) / (n - k))
    V = c * XtX_inv @ meat @ XtX_inv
    se = np.sqrt(np.diag(V))
    t = b / se
    q = _t_q(G - 1)
    return b, se, np.c_[b - q * se, b + q * se], _t_sf2(t, G - 1), G

# ---------------------------------------------------------------------------
# 1. Load
# ---------------------------------------------------------------------------
master    = spark.table(f"{SCHEMA}.master").toPandas()
X_mat_raw = spark.table(f"{SCHEMA}.X_material").toPandas()
X_exp_raw = spark.table(f"{SCHEMA}.X_exp_v2").toPandas()

def is_use_h2(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1"])

master["y_log10_1p_final"] = pd.to_numeric(master["y_log10_1p_final"], errors="coerce")
master["bandgap_eV_meas"]  = pd.to_numeric(master["bandgap_eV"], errors="coerce")
df_base = master[is_use_h2(master["use_h2"]) &
                 master["y_log10_1p_final"].notna() &
                 master["doi_key"].notna() &
                 master["base_compound"].notna()].copy()
df_base["y"] = df_base["y_log10_1p_final"].astype(float)

for t in [df_base, X_mat_raw, X_exp_raw]:
    t["source_row_id"] = t["source_row_id"].astype(str).str.strip()

META = {"source_row_id", "base_compound", "family_proposed", "formula_base_paper",
        "A_site_paper", "B_site_paper", "doi", "doi_key", "y_log10_1p_final",
        "y_log10_1p", "h2_rate_umol_g_h", "h2_umol_h_rate_mass"}
for c in X_mat_raw.columns:
    if c != "source_row_id":
        X_mat_raw[c] = pd.to_numeric(X_mat_raw[c], errors="coerce")
x_mat = [c for c in X_mat_raw.columns if c not in META
         and pd.api.types.is_numeric_dtype(X_mat_raw[c])
         and X_mat_raw[c].isna().mean() < 0.50]

CAT_EXP = ["co_catalyst_type", "run_type", "scavenger_class", "lamp_type", "prep_class"]
x_exp_num = [c for c in X_exp_raw.columns if c not in ["source_row_id"] + CAT_EXP]
for c in x_exp_num:
    X_exp_raw[c] = pd.to_numeric(X_exp_raw[c], errors="coerce")

df = (df_base[["source_row_id", "doi_key", "base_compound", "y", "bandgap_eV_meas"]]
      .merge(X_mat_raw[["source_row_id"] + x_mat], on="source_row_id", how="left")
      .merge(X_exp_raw[["source_row_id"] + x_exp_num + CAT_EXP],
             on="source_row_id", how="left"))
df["base_compound"] = df["base_compound"].astype(str).str.strip()
print(f"Rows: {len(df)} | DOIs: {df['doi_key'].nunique()}")

COMPACT = [c for c in ["bandgap_eV_meas", "ECB_butler", "EVB_butler", "chi_mulliken",
                       "goldschmidt_t", "octahedral_mu", "Ra_Rb_ratio", "sigma_r_A",
                       "sigma_r_B", "wav_gs_bandgap", "wav_en_pauling",
                       "wav_ionic_radius_shannon_vi", "wav_num_d_valence",
                       "wav_mendeleev_number", "wav_first_ion_en",
                       "wav_electron_affinity"] if c in df.columns]
# Within-paper variation in these process/measured variables is NOT a
# composition effect — they are covariates at paper level, never material.
EXP_WITHIN = [c for c in x_exp_num if c in df.columns]

# ---------------------------------------------------------------------------
# 2. Block definitions
# ---------------------------------------------------------------------------
BLOCK_COND = ["run_type", "scavenger_class", "co_catalyst_type", "co_catalyst_loading",
              "cutoff_nm_final", "lamp_power_W", "catalyst_mass_g", "volume_mL",
              "organic_vol_fraction", "T_reaction_C"]
BLOCK_COND = [c for c in BLOCK_COND if c in df.columns]

def cond_str(s):
    if pd.api.types.is_numeric_dtype(s):
        return s.map(lambda v: "na" if pd.isna(v) else f"{v:.4g}")
    return s.astype(str).fillna("na")

df["block_strict"] = df["doi_key"].astype(str)
for c in BLOCK_COND:
    df["block_strict"] = df["block_strict"] + "|" + cond_str(df[c])
df["block_paper"] = df["doi_key"].astype(str)

NUM_FEATS = sorted(set(COMPACT + x_mat + EXP_WITHIN))

def prepare(level):
    """Average per (block, compound); keep blocks with ≥2 distinct compounds."""
    bcol = f"block_{level}"
    agg = (df.groupby([bcol, "base_compound"])
             .agg(doi_key=("doi_key", "first"), y=("y", "mean"),
                  **{c: (c, "mean") for c in NUM_FEATS})
             .reset_index().rename(columns={bcol: "block"}))
    n_comp = agg.groupby("block")["base_compound"].transform("nunique")
    agg = agg[n_comp >= 2].reset_index(drop=True)
    sizes = agg.groupby("block").size()
    print(f"\n[{level}] blocks: {agg['block'].nunique()} | DOIs: {agg['doi_key'].nunique()} | "
          f"material rows: {len(agg)} | compounds/block median {sizes.median():.0f}, "
          f"max {sizes.max()} | within-block SD(y) median "
          f"{agg.groupby('block')['y'].std().median():.2f}")
    return agg

DATA = {"strict": prepare("strict"), "paper": prepare("paper")}

CONFIGS = {
    "strict": {
        "EG_ONLY":  ["bandgap_eV_meas"],
        "COMPACT":  COMPACT,
        "L1_FULL":  x_mat,
    },
    "paper": {
        "EXP_WITHIN":           EXP_WITHIN,
        "EXP_WITHIN+COMPACT":   EXP_WITHIN + COMPACT,
        "EXP_WITHIN+L1_FULL":   EXP_WITHIN + x_mat,
    },
}

# ---------------------------------------------------------------------------
# 3. Helpers
# ---------------------------------------------------------------------------
def demean(values, blocks):
    """Subtract block mean column-wise (works for 1-D and 2-D arrays)."""
    v = pd.DataFrame(values)
    return (v - v.groupby(blocks.values).transform("mean")).values

def pair_accuracy(yc, pc, blocks):
    correct = total = 0.0
    for _, idx in pd.Series(range(len(blocks)), index=blocks.values).groupby(level=0):
        ii = idx.values
        for a in range(len(ii)):
            for b in range(a + 1, len(ii)):
                dy = yc[ii[b]] - yc[ii[a]]
                if abs(dy) < PAIR_MIN:
                    continue
                dp = pc[ii[b]] - pc[ii[a]]
                total += 1
                correct += 0.5 if dp == 0 else float(np.sign(dp) == np.sign(dy))
    return (correct / total if total else np.nan), int(total)

def models():
    return {
        "Ridge":   RidgeCV(alphas=np.logspace(-2, 3, 21)),
        "XGBoost": xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3,
                                    subsample=0.8, colsample_bytree=0.7,
                                    min_child_weight=3, reg_lambda=2.0,
                                    tree_method="hist", random_state=SEED, verbosity=0),
    }

def run_split(tr, va, cols, ycol):
    imp = SimpleImputer(strategy="median")
    Xtr = imp.fit_transform(tr[cols].values.astype(float))
    Xva = imp.transform(va[cols].values.astype(float))
    Xtr, Xva = demean(Xtr, tr["block"]), demean(Xva, va["block"])
    ytr = demean(tr[ycol].values, tr["block"]).ravel()
    yva = demean(va[ycol].values, va["block"]).ravel()
    sc = StandardScaler().fit(Xtr)
    Xtr, Xva = sc.transform(Xtr), sc.transform(Xva)
    out = {}
    for mname, m in models().items():
        m.fit(Xtr, ytr)
        pc = demean(m.predict(Xva), va["block"]).ravel()
        ss_tot = np.sum(yva ** 2)
        r2w = 1 - np.sum((yva - pc) ** 2) / ss_tot if ss_tot > 0 else np.nan
        acc, npairs = pair_accuracy(yva, pc, va["block"].reset_index(drop=True))
        out[mname] = (r2w, acc, npairs)
    return out

# ---------------------------------------------------------------------------
# 4. Repeated grouped CV by DOI
# ---------------------------------------------------------------------------
def fold_assignment(agg, rep):
    dois = np.array(sorted(agg["doi_key"].unique()))
    perm = np.random.default_rng(SEED + rep).permutation(dois)
    fold_of = {d: i % N_FOLDS for i, d in enumerate(perm)}
    return agg["doi_key"].map(fold_of).values

def cv_once(agg, f, cols, ycol):
    """One pass over the 5 folds; returns per-model list of (r2w, acc, npairs)."""
    out = {}
    for k in range(N_FOLDS):
        tr = agg[f != k].reset_index(drop=True)
        va = agg[f == k].reset_index(drop=True)
        assert not set(tr["doi_key"]) & set(va["doi_key"]), "DOI leakage!"
        for mname, vals in run_split(tr, va, cols, ycol).items():
            out.setdefault(mname, []).append((k,) + vals)
    return out

rows = []
for level, agg in DATA.items():
    for rep in range(N_REPEATS):
        f = fold_assignment(agg, rep)
        for name, cols in CONFIGS[level].items():
            for mname, lst in cv_once(agg, f, cols, "y").items():
                for k, r2w, acc, npairs in lst:
                    rows.append({"level": level, "config": name, "model": mname,
                                 "repeat": rep, "fold": k, "r2_within": r2w,
                                 "pair_acc": acc, "n_pairs": npairs})
        print(f"  [{level}] repeat {rep + 1}/{N_REPEATS} done")

res = pd.DataFrame(rows)

# ---------------------------------------------------------------------------
# 4b. Permutation test (y permuted independently inside every block)
# ---------------------------------------------------------------------------
PERM_TARGETS = {"strict": ["EG_ONLY", "COMPACT"], "paper": ["EXP_WITHIN+COMPACT"]}
perm_rows = []
for level, cfgs in PERM_TARGETS.items():
    agg = DATA[level].copy()
    f0 = fold_assignment(agg, 0)
    block_idx = list(agg.groupby("block").indices.values())
    rng_p = np.random.default_rng(SEED + 1000)
    y_orig = agg["y"].values.copy()
    for cfg in cfgs:
        obs = cv_once(agg, f0, CONFIGS[level][cfg], "y")
        null = {m: [] for m in obs}
        for _ in range(N_PERM):
            yp = y_orig.copy()
            for idx in block_idx:
                yp[idx] = y_orig[rng_p.permutation(idx)]
            agg["y_perm"] = yp
            for mname, lst in cv_once(agg, f0, CONFIGS[level][cfg], "y_perm").items():
                null[mname].append((np.nanmean([v[1] for v in lst]),
                                    np.nanmean([v[2] for v in lst])))
        for mname in obs:
            o_r2  = np.nanmean([v[1] for v in obs[mname]])
            o_acc = np.nanmean([v[2] for v in obs[mname]])
            n_r2  = np.array([v[0] for v in null[mname]])
            n_acc = np.array([v[1] for v in null[mname]])
            perm_rows.append({
                "level": level, "config": cfg, "model": mname,
                "obs_pair_acc": o_acc,
                "null_pair_acc_mean": n_acc.mean(),
                "null_pair_acc_p95": np.percentile(n_acc, 95),
                "p_pair_acc": (1 + np.sum(n_acc >= o_acc)) / (1 + N_PERM),
                "obs_r2_within": o_r2,
                "null_r2_p95": np.percentile(n_r2, 95),
                "p_r2_within": (1 + np.sum(n_r2 >= o_r2)) / (1 + N_PERM),
            })
        print(f"  permutation test [{level}] {cfg} done")
perm_df = pd.DataFrame(perm_rows)

def p(q):
    return lambda s: np.nanpercentile(s, q)

summary = (res.groupby(["level", "config", "model"])
              .agg(r2w_mean=("r2_within", "mean"), r2w_p2_5=("r2_within", p(2.5)),
                   r2w_p97_5=("r2_within", p(97.5)),
                   pair_acc_mean=("pair_acc", "mean"), pair_p2_5=("pair_acc", p(2.5)),
                   pair_p97_5=("pair_acc", p(97.5)),
                   pairs_per_split=("n_pairs", "mean"))
              .reset_index().sort_values(["level", "pair_acc_mean"], ascending=[False, False]))
pd.set_option("display.width", 220)
print(f"\n--- Within-block performance, {N_REPEATS}×{N_FOLDS} GroupKFold by DOI ---")
print(summary.round(3).to_string(index=False))
print("\nNull references: r2_within = 0, pair_acc = 0.5. The p2.5–p97.5 band is the "
      "spread over re-splits of ONE dataset — it is NOT a confidence interval.")

print(f"\n--- Permutation test (repeat-0 splits, {N_PERM} within-block permutations) ---")
print(perm_df.round(3).to_string(index=False))
print("  p < 0.05 → material ranking signal beyond chance under fixed conditions.")

# ---------------------------------------------------------------------------
# 5. Inference: within-block OLS with cluster-robust SE (strict level)
# ---------------------------------------------------------------------------
INF_FEATS = [c for c in ["bandgap_eV_meas", "chi_mulliken", "goldschmidt_t",
                         "octahedral_mu", "wav_num_d_valence"] if c in DATA["strict"].columns]
for level in ["strict", "paper"]:
    agg = DATA[level]
    cc = agg.dropna(subset=INF_FEATS + ["y"]).copy()
    cc = cc[cc.groupby("block")["base_compound"].transform("nunique") >= 2]
    if cc["block"].nunique() < 10:
        print(f"\n[{level}] too few complete-case blocks for inference "
              f"({cc['block'].nunique()}) — skipped.")
        continue
    Z = (cc[INF_FEATS] - cc[INF_FEATS].mean()) / cc[INF_FEATS].std()
    Xw = demean(Z.values, cc["block"])
    yw = demean(cc["y"].values, cc["block"]).ravel()
    keep = np.nanstd(Xw, axis=0) > 0
    b, se, ci, pvals, G = ols_cluster(yw, Xw[:, keep],
                                      pd.factorize(cc["doi_key"])[0])
    print(f"\n--- [{level}] within-block OLS, cluster-robust (CR1) by DOI "
          f"({len(cc)} material rows, {cc['block'].nunique()} blocks, "
          f"{G} DOIs) ---")
    print("  Coefficient = Δ log10(rate) per +1 SD of descriptor, conditions fixed")
    k_tests = int(keep.sum())
    for name, bi, lo, hi, pv in zip(np.array(INF_FEATS)[keep], b,
                                    ci[:, 0], ci[:, 1], pvals):
        p_bonf = min(1.0, pv * k_tests)
        flag = "  *" if p_bonf < 0.05 else ""
        print(f"  {name:22s} {bi:+.3f}  [{lo:+.3f}, {hi:+.3f}]  "
              f"p={pv:.3f}  p_bonf={p_bonf:.3f}{flag}")
    print(f"  * = significant after Bonferroni correction for {k_tests} tests.")

# ---------------------------------------------------------------------------
# 6. Plot + write
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, level in zip(axes, ["strict", "paper"]):
    s = summary[summary["level"] == level].sort_values("pair_acc_mean")
    labels = s["config"] + " | " + s["model"]
    ax.errorbar(s["pair_acc_mean"], range(len(s)),
                xerr=[s["pair_acc_mean"] - s["pair_p2_5"], s["pair_p97_5"] - s["pair_acc_mean"]],
                fmt="o", capsize=3)
    ax.axvline(0.5, color="black", lw=0.8)
    ax.set_yticks(range(len(s))); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Pairwise ranking accuracy (null = 0.5)")
    ax.set_title(f"{level} blocks")
plt.tight_layout()
plt.savefig("/tmp/nb13_within.png", dpi=120)
plt.close()
print("\nSaved: /tmp/nb13_within.png")

spark.createDataFrame(res).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.within_nb13")
spark.createDataFrame(perm_df).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.within_perm_nb13")
print(f"✓ Written: {SCHEMA}.within_nb13 ({len(res)} rows)")
print(f"✓ Written: {SCHEMA}.within_perm_nb13 ({len(perm_df)} rows)")
print("\n=== NB13 COMPLETE ===")